# Analiza EEG z MNE-Python: notatnik do prezentacji
Komórki kodu są identyczne z fragmentami na slajdach (`presentation/code/sNN.py`) i wykonują się po kolei we wspólnej przestrzeni nazw. Dane (PhysioNet EEGBCI, uczestnik 1, biegi 6, 10, 14) pobiera `eegbci.load_data()`. Katalog danych: zmienna `MNE_DATA` (domyślnie `data/`).

In [1]:
import os, sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
os.environ.setdefault("MNE_DATA", str(ROOT / "data"))
os.environ.setdefault("MNE_DATASETS_EEGBCI_PATH", os.environ["MNE_DATA"])

'/home/claude/mne-eeg-presentation/data'

## Slajd 06. Import bibliotek

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import mne
from mne.datasets import eegbci
from mne.io import (concatenate_raws,
                    read_raw_edf)

mne.set_log_level("WARNING")
print("MNE", mne.__version__)
print("NumPy", np.__version__)


MNE 1.13.2
NumPy 2.5.4


## Slajd 07. Pobieranie danych

In [3]:
files = eegbci.load_data(1, [6, 10, 14],
                         update_path=False)
for f in files:
    print(f.name, f.stat().st_size // 1000, "kB")


S001R06.edf 2596 kB
S001R10.edf 2596 kB
S001R14.edf 2596 kB


/usr/local/lib/python3.12/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Slajd 08. Wczytywanie plików EDF

In [4]:
raws = [read_raw_edf(f, preload=True)
        for f in files]
lens = [int(r.n_times) for r in raws]
raw = concatenate_raws(raws)
eegbci.standardize(raw)
print(lens, raw.n_times)
print(raw.ch_names[:5])


[20000, 20000, 20000] 60000
['FC5', 'FC3', 'FC1', 'FCz', 'FC2']


## Slajd 09. Metadane EEG

In [5]:
info = raw.info
fs = info["sfreq"]
print("sfreq:", fs, "| nchan:", info["nchan"])
print("czas:", raw.times[-1] + 1 / fs, "s")
cz = raw.get_data(picks="Cz")[0]
print("Cz [V]:", cz[:3])
print("Cz [uV]:", cz[:3] * 1e6)


sfreq: 160.0 | nchan: 64
czas: 375.0 s
Cz [V]: [8.2e-05 7.7e-05 6.5e-05]
Cz [uV]: [82. 77. 65.]


## Slajd 10. Rozmieszczenie elektrod

In [6]:
raw.set_montage("colin27_1005")
pos = raw.get_montage().get_positions()
print(len(pos["ch_pos"]), "pozycji w montazu")
print("Cz [m]:", pos["ch_pos"]["Cz"].round(3))
print("C3 [m]:", pos["ch_pos"]["C3"].round(3))


64 pozycji w montazu
Cz [m]: [-0.001  0.028  0.14 ]
C3 [m]: [-0.067  0.023  0.105]


## Slajd 11. Surowy sygnał EEG

In [7]:
seg = raw.copy().crop(20, 30)
chs = ["Fp1", "C3", "Cz", "C4", "Oz"]
data = seg.get_data(picks=chs, units="uV")
print(data.shape, np.ptp(data[1]).round(1))


(5, 1601) 419.0


## Slajd 12. Znaczniki eksperymentalne

In [8]:
ann = raw.annotations
desc = [d for d in ann.description
        if d[0] == "T"]
cnt = [desc.count(k) for k in ("T0","T1","T2")]
print("T0/T1/T2:", *cnt)


T0/T1/T2: 45 21 24


## Slajd 13. Widmo mocy EEG

In [9]:
spec = raw.compute_psd(
    method="welch", fmax=45, n_fft=640)
psd = spec.get_data(picks="C3")[0]
mu = (spec.freqs >= 8) & (spec.freqs <= 13)
print("C3 mu:", (psd[mu] * 1e12).mean().round(1))


C3 mu: 43.5


## Slajd 14. Filtracja EEG

In [10]:
raw_tfr = raw.copy().filter(
    1.0, 40.0, fir_design="firwin")
raw_clf = raw.copy().filter(
    7.0, 30.0, fir_design="firwin")
print(raw_clf.info["lowpass"], "Hz")


30.0 Hz


## Slajd 15. Referencjonowanie EEG

In [11]:
for r in (raw_tfr, raw_clf):
    r.set_eeg_reference("average",
                        projection=False)
x = raw_clf.get_data()
print("suma = 0:", abs(x.sum(0)).max() < 1e-12)


suma = 0: True


## Slajd 16. Wykrywanie zdarzeń

In [12]:
events, _ = mne.events_from_annotations(
    raw_tfr, event_id={"T1": 1, "T2": 2})
print(events.shape)
print(events[:4])
print(np.bincount(events[:, 2])[1:])


(45, 3)
[[ 672    0    2]
 [2000    0    1]
 [3328    0    1]
 [4656    0    2]]
[21 24]


## Slajd 17. Segmentacja EEG

In [13]:
epochs = mne.Epochs(
    raw_tfr, events, dict(hands=1, feet=2),
    tmin=-1.5, tmax=4.5, baseline=None,
    reject=None, preload=True)
print(epochs.get_data().shape)


(45, 64, 961)


## Slajd 18. Kontrola jakości

In [14]:
import scipy.stats as st
pp = np.ptp(epochs.get_data(), axis=2).max(1)
mad = st.median_abs_deviation(pp, scale="normal")
thr = np.median(pp) + 3 * mad
print("200 uV:", (pp > 200e-6).sum(), "z", len(pp))
epochs.drop_bad(reject=dict(eeg=thr))
print(round(thr * 1e6), "uV;", len(epochs), "epok")


200 uV: 43 z 45
722 uV; 44 epok


## Slajd 19. Independent Component Analysis

In [15]:
ica = mne.preprocessing.ICA(n_components=20,
                            random_state=97)
ica.fit(raw_tfr)
idx, sc = ica.find_bads_eog(raw_tfr, ch_name="Fp1")
print("kandydaci:", list(map(int, idx)))


kandydaci: [0]


## Slajd 20. Uśrednianie EEG

In [16]:
evokeds = {c: epochs[c].average()
           for c in ["hands", "feet"]}
print([e.nave for e in evokeds.values()])


[21, 23]


## Slajd 21. Analiza czasowo-częstotliwościowa

In [17]:
freqs = np.arange(6, 36, 1.0)
tfr = {c: epochs[c].compute_tfr(
           "morlet", freqs, n_cycles=freqs / 2,
           decim=2, average=True)
       for c in ["hands", "feet"]}
print(tfr["hands"].data.shape)


(64, 30, 481)


## Slajd 22. Normalizacja i ERD/ERS

In [18]:
erd = {c: t.copy().apply_baseline(
           (-0.8, -0.2), mode="percent")
       for c, t in tfr.items()}
win = dict(fmin=8, fmax=13, tmin=1, tmax=3)
v = erd["hands"].get_data(picks=["C3"], **win)
print("C3, hands: %.0f %%" % (100 * v.mean()))


C3, hands: -55 %


## Slajd 23. Porównanie warunków eksperymentalnych

In [19]:
for ch in ["C3", "Cz", "C4"]:
    h, f = [100 * erd[c].get_data(
            picks=[ch], **win).mean()
            for c in ("hands", "feet")]
    print(ch, round(h), round(f))


C3 -55 12
Cz -18 19
C4 -45 18


## Slajd 24. Analiza pasma μ

In [20]:
mu = {c: 100 * erd[c].get_data(
          **win).mean((1, 2))
      for c in ["hands", "feet"]}
for c, v in mu.items():
    print(c, round(v.min()), round(v.max()))


hands -55 42
feet -16 74


## Slajd 26. Przygotowanie danych ML

**Instrukcja:** najpierw zobacz `X.shape`, potem przeczytaj `y` i `groups`. Wiersz `X` to jedna próba: 64 kanały oraz próbki z okresu 0,5–3,5 s po wskazówce. `y` to znane z eksperymentu etykiety `hands`/`feet`, a `groups` to identyfikatory trzech biegów. Te grupy zabezpieczą test przed umieszczeniem prób tego samego biegu po obu stronach podziału.


In [21]:
epochs_clf = mne.Epochs(
    raw_clf, epochs.events,
    dict(hands=1, feet=2),
    tmin=-1.5, tmax=4.5, baseline=None,
    reject_by_annotation=False, preload=True)
X = epochs_clf.get_data(tmin=0.5, tmax=3.5)
y = epochs_clf.events[:, 2]
t0 = epochs_clf.events[:, 0] - raw.first_samp
edges = np.cumsum(lens)
groups = np.searchsorted(edges, t0, side="right")
print(X.shape, np.bincount(y)[1:],
      np.bincount(groups))


(44, 64, 480) [21 23] [15 14 15]


## Slajd 27. Klasyfikator CSP + LDA

**Co to jest klasyfikator?** To algorytm przypisujący nieznanej epoce EEG jedną z dwóch klas. Nasz cel to rozpoznać z sygnału, czy w danej próbie wyobrażano ruch obu rąk, czy obu stóp. CSP uczy się z danych treningowych filtrów przestrzennych i przekształca epokę w cztery logarytmy wariancji. LDA szacuje granicę rozdzielającą klasy i przewiduje etykietę. Zarówno CSP, jak i LDA muszą być trenowane na każdym foldzie osobno. Model nie diagnozuje funkcji ruchowych ani nie wykrywa dowolnych intencji.


In [22]:
from mne.decoding import CSP
from sklearn.discriminant_analysis import (
    LinearDiscriminantAnalysis as LDA)
from sklearn.pipeline import make_pipeline

clf = make_pipeline(
    CSP(n_components=4, reg="ledoit_wolf",
        log=True),
    LDA())
print([name for name, _ in clf.steps])


['csp', 'lineardiscriminantanalysis']


## Slajd 28. Walidacja modelu

**Jak czytać:** każdy fold ma dwa biegi do uczenia i jeden, dotąd niewidziany przez CSP i LDA, do oceny. `balanced_accuracy` to średnia trafność rozpoznawania obu klas, odpowiednia również przy nierównych liczbach prób. Porównanie z losowym podziałem epok ma charakter dydaktyczny. To test generalizacji między biegami **tej samej osoby**, nie między osobami.


In [23]:
import sklearn.model_selection as ms
cv = ms.GroupKFold(n_splits=3)
bal = dict(scoring="balanced_accuracy")
scores = ms.cross_val_score(
    clf, X, y, groups=groups, cv=cv, **bal)
skf = ms.StratifiedKFold(
    3, shuffle=True, random_state=0)
rnd = ms.cross_val_score(
    clf, X, y, cv=skf, **bal)
print("biegi :", scores.round(2))
print("losowo:", rnd.round(2))


biegi : [0.93 1.   1.  ]
losowo: [0.88 0.93 1.  ]


## Slajd 29. Ocena jakości klasyfikacji

**Po co drugi model?** Klasyfikator większościowy zawsze wskazuje częstszą klasę; dla dwóch obecnych klas jego trafność zrównoważona wynosi około 0,5. Macierz pomyłek pozwala sprawdzić, które próby pomylono. Wynik CSP+LDA oceniany jest na predykcjach otrzymanych z modeli, które nie widziały testowanych biegów. Wniosku nie wolno przenosić na innych uczestników bez osobnego eksperymentu.


In [24]:
import sklearn.metrics as sm
from sklearn.dummy import DummyClassifier
bas = sm.balanced_accuracy_score
y_hat = ms.cross_val_predict(
    clf, X, y, groups=groups, cv=cv)
ba = bas(y, y_hat)
cm = sm.confusion_matrix(y, y_hat)
dummy = DummyClassifier(
    strategy="most_frequent")
y_base = ms.cross_val_predict(
    dummy, X, y, groups=groups, cv=cv)
print(round(ba, 3), round(bas(y, y_base), 3))


0.976 0.5


## Dodatek: test permutacyjny (ok. 1,5 min)

**Ograniczenie:** test permutacyjny miesza etykiety wyłącznie w obrębie biegów i wymaga założenia wymienności prób pod hipotezą zerową. Niesuperwizowany próg QC ustalono wcześniej z całego sygnału, dlatego nie traktujemy tej procedury jako ścisłej walidacji end-to-end.


In [25]:
y_hat = ms.cross_val_predict(
    clf, X, y, groups=groups, cv=cv)
s, perm, p = ms.permutation_test_score(
    clf, X, y, groups=groups, cv=cv, random_state=0,
    scoring="balanced_accuracy", n_permutations=200)
print("p =", round(p, 3), "| srednia perm.:", perm.mean().round(2))


p = 0.005 | srednia perm.: 0.49


## Slajd 30. Interpretacja komponentów CSP

In [26]:
csp = CSP(n_components=4, reg="ledoit_wolf",
          log=True)
csp.fit(X, y)
print(csp.patterns_.shape, csp.filters_.shape)


(63, 64) (63, 64)


## Slajd 31. Eksport wyników

In [27]:
import json, sklearn
from pathlib import Path
out = Path("output/results")
out.mkdir(parents=True, exist_ok=True)
epochs_clf.save(out / "S001-clf-epo.fif",
                overwrite=True)
res = dict(subject=1, n_epochs=len(y),
           bal_acc=round(ba, 3),
           mne=mne.__version__,
           sklearn=sklearn.__version__)
f = out / "results.json"
f.write_text(json.dumps(res))
print(sorted(p.name for p in out.iterdir()))


['S001-clf-epo.fif', 'results.json']


## Slajd 32. Podsumowanie i zadanie praktyczne

In [28]:
import sys
sys.path.insert(0, "python")
from analysis import run_subject

res = run_subject(subject=1)
print(res["n_epochs"], round(res["bal_acc"], 3))
# zadanie: subject=2


44 0.976


## Ćwiczenie
1. `run_subject(subject=2)` (walidacja w obrębie osoby).
2. Trening na S1, test na S2.
3. Porównaj wyniki; parametry wybieraj bez użycia zbioru testowego.